# Phase 4 — Metric Exploration

Use this notebook to explore the current report data, validate signal distributions,
spot outliers, and research new metrics before adding them to the pipeline.

**Workflow:**
1. Explore/validate here first
2. Once confident, add to `pipeline/value_metrics.py` or `pipeline/fraud_signals.py`
3. Regenerate report with `python3 run.py --signals`
4. Display in `app.py`

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')

# Load the latest report
with open('../reports/fraud_report.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} companies")
print(f"Columns: {len(df.columns)}")
df.head(3)

## Signal coverage — how many companies have each metric?

In [ ]:
# Coverage report for all numeric columns
numeric_cols = df.select_dtypes(include=[float, int]).columns
coverage = (
    df[numeric_cols].notna().sum()
    .sort_values(ascending=False)
    .reset_index()
)
coverage.columns = ['metric', 'count']
coverage['coverage_%'] = (coverage['count'] / len(df) * 100).round(1)
coverage

## Market cap segment distribution

In [ ]:
seg_order = ['micro', 'small', 'mid', 'large']
seg_counts = df['market_cap_segment'].value_counts().reindex(seg_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 4))
seg_counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
ax.set_title('Companies by Market Cap Segment')
ax.set_ylabel('Count')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(seg_counts.to_string())

## Fraud score distribution by segment

Key question: do small/micro caps have higher fraud scores on average?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = {'micro': '#e74c3c', 'small': '#f39c12', 'mid': '#3498db', 'large': '#2ecc71'}

for ax, seg in zip(axes, seg_order):
    subset = df[df['market_cap_segment'] == seg]['fraud_score'].dropna()
    ax.hist(subset, bins=30, color=colors[seg], alpha=0.8, edgecolor='none')
    ax.set_title(f'{seg.title()} (n={len(subset)})')
    ax.set_xlabel('Fraud Score')
    ax.axvline(70, color='red', linestyle='--', alpha=0.5, label='High risk')
    ax.axvline(45, color='orange', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Count')
plt.suptitle('Fraud Score Distribution by Segment', y=1.02)
plt.tight_layout()
plt.show()

## Magic Formula — top candidates

Low MF rank + low fraud score = the sweet spot (cheap + quality + clean)

In [ ]:
mf = df[df['magic_formula_rank'].notna()].copy()
mf = mf.sort_values('magic_formula_rank')

# Top 20 Magic Formula — with fraud score as secondary filter
cols = ['ticker', 'name', 'market_cap_segment', 'fraud_score', 'risk',
        'magic_formula_rank', 'earnings_yield', 'return_on_capital',
        'acquirers_multiple', 'gross_profitability']
mf[cols].head(20).style.background_gradient(subset=['fraud_score'], cmap='RdYlGn_r')

## Alpha candidates: Cheap + Survive + Avoid fraud

Intersection of: top MF rank quartile + low fraud score + not distress zone

In [ ]:
mf_threshold = df['magic_formula_rank'].quantile(0.25)  # top 25%

candidates = df[
    (df['magic_formula_rank'] <= mf_threshold) &
    (df['fraud_score'] < 45) &                          # low fraud risk
    (df['altman_zone'].isin(['safe', 'grey'])) &         # not distress
    (df['going_concern_flag'] == False)
].copy()

candidates = candidates.sort_values('magic_formula_rank')
print(f"Alpha candidates: {len(candidates)}")
candidates[cols].head(30).style.background_gradient(subset=['fraud_score'], cmap='RdYlGn_r')

## Net-net companies (Graham liquidation floor)

Market cap < NCAV = trading below liquidation value

In [ ]:
net_nets = df[df['net_net_flag'] == True].copy()
net_nets = net_nets.sort_values('ncav_ratio')
print(f"Net-net companies: {len(net_nets)}")

nn_cols = ['ticker', 'name', 'market_cap_segment', 'fraud_score', 'risk',
           'ncav_ratio', 'ncav', 'market_cap', 'current_ratio', 'going_concern_flag']
net_nets[nn_cols].head(20)

## Signal correlation heatmap

Which signals move together? High correlation = redundancy.

In [ ]:
signal_cols = [
    'fraud_score', 'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'non_op_ratio',
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'volatility_90d', 'beta'
]
existing = [c for c in signal_cols if c in df.columns]
corr = df[existing].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(existing)))
ax.set_yticks(range(len(existing)))
ax.set_xticklabels(existing, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(existing, fontsize=9)
ax.set_title('Signal Correlation Matrix')
plt.tight_layout()
plt.show()